In [1]:
import os
import pandas as pd
import networkx as nx
from functools import partial
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.graphs import construct_graph
from graphein.protein.visualisation import plotly_protein_structure_graph
from graphein.protein.edges.distance import add_distance_threshold

In [2]:


percorso_indice = "data_pdbbind/index/INDEX_general_PL.2020R1.lst"
percorso_cartelle_3d = "data_pdbbind/archive"

# 2. Leggiamo l'Indice ufficiale
# Il file di PDBbind è un po' fastidioso: usa spazi multipli e ha commenti con il simbolo #
nomi_colonne = ['PDB_code', 'resolution', 'release_year', 'pKd_pKi', 'ligand_name']

print("Leggo l'indice generale di PDBbind...")
df_indice = pd.read_csv(
    percorso_indice,
    sep='\s+',              # Significa: usa uno o più spazi come separatore di colonna
    comment='#',            # Ignora tutta l'introduzione di testo che inizia con #
    header=None,            # Diciamo a Pandas che non c'è una riga di intestazione pulita
    names=nomi_colonne,     # Assegniamo noi i nomi giusti alle colonne
    usecols=[0, 1, 2, 3, 4] # Prendiamo solo le prime 5 colonne (le altre sono info extra che non ci servono)
)
print(f"✅ Fatto! Trovati {len(df_indice)} complessi nell'indice ufficiale.")

# 3. Vediamo quante cartelle (molecole) hai FISICAMENTE sul tuo PC
print("\nCerco le cartelle 3D sul tuo computer...")
cartelle_locali = [nome for nome in os.listdir(percorso_cartelle_3d) if os.path.isdir(os.path.join(percorso_cartelle_3d, nome))]
print(f"✅ Trovate {len(cartelle_locali)} cartelle 3D locali.")

# 4. Il Match Magico: Filtriamo il DataFrame
df_locale = df_indice[df_indice['PDB_code'].isin(cartelle_locali)].copy()
print(f"\n🎉 MATCH COMPLETATO! Abbiamo i dati completi per {len(df_locale)} complessi.")

# 5. Salviamo questo "mini-indice" pulito, così non dobbiamo rifare questa cosa ogni volta
percorso_salvataggio = "data_pdbbind/mio_indice_locale.csv"
df_locale.to_csv(percorso_salvataggio, index=False)
print(f"File salvato in: {percorso_salvataggio}")

# Mostriamo le prime 5 righe per assicurarci che sia tutto perfetto
df_locale.head()

Leggo l'indice generale di PDBbind...
✅ Fatto! Trovati 19037 complessi nell'indice ufficiale.

Cerco le cartelle 3D sul tuo computer...
✅ Trovate 195 cartelle 3D locali.

🎉 MATCH COMPLETATO! Abbiamo i dati completi per 195 complessi.
File salvato in: data_pdbbind/mio_indice_locale.csv


,PDB_code,resolution,release_year,pKd_pKi,ligand_name
8,4tmn,1.70,1989,Ki=0.068nM,//
64,1igj,2.50,1993,Kd=0.1nM,//
320,1kel,1.90,1996,Kd=52nM,//
330,1sln,2.27,1996,Ki=0.23uM,//
442,2gss,1.90,1997,Ki=11.5uM,//


In [ ]:


#primo PDB ID dal dataframe locale
pdb_id = df_locale['PDB_code'].iloc[0]
print(f"Proteina selezionata: {pdb_id}")

#creazione e visualizzazione del grafo
def create_protein_graph(pdb_code: str) -> nx.Graph:
    edge_funcs = [partial(add_distance_threshold, long_interaction_threshold=0, threshold=8.0)]
    config = ProteinGraphConfig(edge_construction_functions=edge_funcs)
    print(f"Costruzione grafo 3D per {pdb_code} in corso...")
    return construct_graph(config=config, pdb_code=pdb_code)

def visualize_protein_3D(g: nx.Graph):
    fig = plotly_protein_structure_graph(
        g,
        colour_edges_by="kind",
        colour_nodes_by="seq_position",
        node_size_multiplier=1
    )
    fig.show()


G_protein = create_protein_graph(pdb_id)
print(f"\nNodi (Amminoacidi): {G_protein.number_of_nodes()}")
print(f"Archi (Interazioni): {G_protein.number_of_edges()}")
visualize_protein_3D(G_protein)

Proteina selezionata: 4tmn
Costruzione grafo 3D per 4tmn in corso...


Output()


Nodi (Amminoacidi): 316
Archi (Interazioni): 2010
